<a href="https://colab.research.google.com/github/tnehezd/mesalab/blob/main/docs/colab_notebooks/mesalab_mesa_grid_base_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Tutorial: Running an Example Analysis in `mesalab` with Google Colab


This notebook will guide you through running a full `mesalab` analysis within a Google Colab environment. We'll set up `mesalab`, process a sample of real MESA data to identify specific stellar phenomena, and inspect the results.

The provided example data allows you to test `mesalab`'s core functionality for filtering MESA outputs **without** needing to install the MESA SDK, GYRE, or RSP.

----

### Prerequisites

* An active Google account (for Colab access).

----


> 💡 **Note** The pipeline is optimized for Python < 3.12! If you encounter any runtime errors, please check that your **Runtime version is set to 2025.07** under the **Notebook settings / Runtime type**. You can find this either in the *Edit -> Notebook settings* menu, or in the *Runtime -> Change runtime type* menu.

## 1.  Set up `mesalab` and Get Example Data

#### 1.1 Clone the ``mesalab`` repository

First, clone the `mesalab` repository from GitHub. This will give you access to the source code and the `example/` directory containing the sample MESA data and configuration files.

> 💡 **Note:** This step is a one-time operation per session.
> If you've already run this cell, you can skip it.


In [ ]:
!git clone https://github.com/konkolyseismolab/mesalab

Cloning into 'mesalab'...
remote: Enumerating objects: 18410, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 18410 (delta 74), reused 93 (delta 39), pack-reused 18254 (from 3)
Receiving objects: 100% (18410/18410), 866.37 MiB | 17.11 MiB/s, done.
Resolving deltas: 100% (9611/9611), done.
Updating files: 100% (950/950), done.


----

#### 1.2 Install `mesalab` and its dependencies

Now, install `mesalab` and all its required Python packages.

> 💡 **Note:** This step is a one-time operation per session.
> If you've already run this cell, you can skip it.


> 💡 **Note on Environment & Dependencies:** > Since Python 3.11/3.12+ is pre-installed on modern Colab environments with a rigid AI/Deep Learning stack, we must force-reinstall specific legacy versions of the scientific computing core (`numpy==1.24.4`, `pandas==2.0.3`, `numba==0.57.1`) required by the MESA/GYRE analysis routines.
>
> ⚠️ **Regarding Dependency Resolver Errors:** > At the end of the installation step, `pip` will display several red `ERROR` messages regarding dependency conflicts (e.g., with `google-colab`, `jax`, or `tensorflow`). **These are internal Colab system warnings and can be safely ignored.** They do not affect `mesalab` in any way, as our code runs directly from the cloned repository using its own isolated, forced runtime environment.


In [31]:
import os
import sys
import shutil

print("======================================================================")
print("1. STEP: Installing underlying astronomy and plotting stack...")
print("======================================================================")
!pip install numpy==1.24.4 pandas==2.0.3 scipy==1.10.1 numba==0.57.1 --force-reinstall --quiet
!pip install addict f90nml configobj isochrones emcee pygyre swifter astropy==5.3.4 seaborn==0.12.2 corner --no-deps --quiet

print("\n======================================================================")
print("2. STEP: Hard-purging corrupted isochrones cache...")
print("======================================================================")

# This physically deletes the entire grid cache folder to get rid of 'truncated header' files
cache_dir = os.path.expanduser('~/.isochrones')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("[CLEANED] All old, corrupted cache files have been forcefully removed.")
else:
    print("Cache directory is clean.")

print("\n======================================================================")
print("3. STEP: Pre-downloading clean MIST Bolometric Correction grids...")
print("======================================================================")

# Set path mappings
cloned_repo_path = '/content/mesalab'
sys.path.insert(0, cloned_repo_path)
os.environ['PYTHONPATH'] = cloned_repo_path

# Force a clean, isolated download of the primary Gaia bands directly via python CLI
print("Downloading clean MIST tables from Harvard servers directly to Colab...")
!python3 -c "from isochrones.mist.bc import MISTBolometricCorrectionGrid; print('Starting download...'); grid = MISTBolometricCorrectionGrid(['G', 'BP', 'RP']); print('[SUCCESS] Grid successfully downloaded and extracted!')"

1. STEP: Installing underlying astronomy and plotting stack...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.0.3 which is incompatible.
blosc2 3.5.1 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
cudf-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.57.1 which is incompatible.
cuml-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.57.1 which is incompatible.
cvxpy 1.6.6 requires scipy>=1.11.0, but you have scipy 1.10.1 which is incompatible.
dask-cuda 25.2.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.57.1 which is incompatible.
distributed-ucxx-cu12 0.42.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.57.1 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
jax 0.5.

----

## 2. Examine the Example Data and Configuration


Once you have successfully installed the package, navigate to the `example/` directory within `mesalab/`.

In [32]:
# Navigate to the example track
%cd /content/mesalab/example


/content/mesalab/example


This directory contains two pre-defined datasets. In this tutorial, we will focus on the `MESA_grid` dataset, which consists of real stellar evolution outputs from MESA. It is designed to demonstrate `mesalab`'s core blue loop filtering and analysis capabilities.

#### 2.1. Dataset Overview

* **Grid Parameters**: The dataset includes a 2x2 grid of models with masses of 4 and 5 M⊙​ and metallicities (Z) of 0.0090 and 0.0100.

* **Evolutionary Coverage**: The simulations cover stellar evolution from the pre-main sequence (pre-MS) to a point after the blue loop phase.

* **Key Feature**: A defining characteristic of this dataset is the differing blue loop behavior: models with 5 M⊙​ exhibit blue loop crossings, while models with 4 M⊙​ do not.


This example dataset is located in the `example/MESA_grid` directory.

The corresponding `example_MESA_base.yaml` configuration file is set up to identify blue loop crossers and generate plots. It also prepares filtered output files, which can be used as input for a subsequent GYRE workflow.

## 3. Run the `MESA_grid` Example

You can easily run your first example by executing `mesalab` with the provided configuration file:

In [33]:
# Force matplotlib to headless mode for cloud rendering
import matplotlib
matplotlib.use('Agg')

# Run the local evolutionary track analyzer directly
!mesalab --config example_MESA_base.yaml


                    mesalab CLI - Starting Analysis Workflow                    
                                 Version: 2.2.0                                 

2026-06-02 12:49:55,504 - DEBUG: Debug mode enabled after full config merge in config_parser.
2026-06-02 12:49:55,504 - INFO: Final resolved configuration: {'general_settings': {'input_dir': 'MESA_grid', 'output_dir': 'MESA_grid_base_output', 'inlist_name': 'inlist_project', 'force_reanalysis': True, 'debug': True, 'mesasdk_root': None, 'mesa_dir': None, 'mesa_binary_dir': None, 'gyre_dir': None}, 'blue_loop_analysis': {'analyze_blue_loop': True, 'blue_loop_output_type': 'summary'}, 'plotting_settings': {'generate_heatmaps': True, 'generate_hr_diagrams': 'all', 'generate_blue_loop_plots_with_bc': True, 'generate_plots': True}, 'gyre_workflow': {'run_gyre_workflow': False, 'gyre_inlist_template_path': 'config/gyre.in', 'run_mode': 'ALL_PROFILES', 'num_gyre_threads': 1, 'enable_gyre_parallel': False, 'max_concurrent_gyre_runs'

! ls


----

#### 3.1. Checking the Ouput

After a successful run, you will find the generated plots in the `example/MESA_grid_base_output/plots` directory. Here are some examples of the plots generated for this grid:


In [30]:
from IPython.display import Image
Image(filename='MESA_grid_base_output/plots/CMD_Gaia_all_blue_loop_data.png')

FileNotFoundError: [Errno 2] No such file or directory: 'MESA_grid_base_output/plots/CMD_Gaia_all_blue_loop_data.png'

**Figure 1:** Gaia Color-Magnitude Diagram (CMD) for the 5 Msun models that undergo blue loop evolution. This plot specifically focuses on models that are currently within the blue loop phase and have crossed the red (cool) boundary of the Instability Strip (IS), indicating evolutionary stages relevant for pulsating stars.

In [ ]:
Image(filename='MESA_grid_base_output/plots/mesa_grid_blue_loop_heatmap.png')

**Figure 2:** Heatmap visualizing the number of instability strip crossings for different initial masses and metallicities.

----

#### 3.2. Additional Plots and CSVs


You can find more plots and CSV files in the `example/MESA_grid_base_output/` directory. These include HR diagrams for each metallicity and a color-magnitude diagram (CMD) of the blue loop evolutionary tracks.